# M20a - Structural path comparison

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Submission evidence.** Allocation temperature, gain, leak rate, and hard top-k sparsity are compared under one current executable protocol.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)
def bootstrap_mean(x,n_boot=30000,seed=1):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(x),size=(n_boot,len(x)))
    b=x[idx].mean(axis=1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))

In [2]:
SEED=20260718; TRIALS=4
TASKS=['controlled_d1_clean','controlled_d20_clean','controlled_d20_white_plus_distractor','memory_d10','narma10','mackey_glass','lorenz_x']
PATHS=['temperature','gain','leak','sparsity']
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=.8,ridge=1e-5)
rep=tcr.run_panel(TASKS,TRIALS,PATHS,seed=SEED,**CONFIG); rep.to_csv(REPRO/'m20a_replication_case_metrics.csv',index=False)
rows=[]
for path,g in rep.groupby('path',sort=False):
    rows.append({'path':path,'n_cases':len(g),'mean_safe_width':g.safe_width.mean(),'safe_gain':g.safe_gain.mean(),'full_gain':g.full_gain.mean(),'safe_minus_matched_gain':g.safe_minus_matched_gain.mean(),'near_containment':g.near_contained.mean(),'matched_near_rate':g.matched_near_rate.mean(),'support_range':g.support_range.mean(),'spectral_radius_range':g.spectral_radius_range.mean()})
summary=pd.DataFrame(rows); summary.to_csv(REPRO/'m20a_replication_summary.csv',index=False)
t=rep[rep.path=='temperature']; ci=bootstrap_mean(t.safe_gain,seed=2001); cm=bootstrap_mean(t.safe_minus_matched_gain,seed=2002)
print('temperature safe-gain CI:',ci); print('temperature safe-control CI:',cm); display(summary.round(6))

temperature safe-gain CI: (0.002532624170134093, 0.0006788711703743544, 0.004960955327364268)
temperature safe-control CI: (0.027508273150726025, 0.018784048471596964, 0.03673836362230405)


,path,n_cases,mean_safe_width,safe_gain,full_gain,safe_minus_matched_gain,near_containment,matched_near_rate,support_range,spectral_radius_range
0,temperature,28,2.892857,0.002533,0.007031,0.027508,0.857143,0.362718,55.16451,0.171865
1,gain,28,3.964286,0.004235,0.010422,0.034531,0.857143,0.496681,0.00000,1.569810
2,leak,28,4.857143,0.004241,0.005015,0.061486,0.964286,0.623177,0.00000,0.000000
3,sparsity,28,1.285714,0.001973,0.009594,0.038468,0.750000,0.238141,58.00000,0.151554
